# Fitting a Lensed SN Ia with bayesn-td

This tutorial demonstrates how to use bayesn-td to fit a multiply-imaged, strongly-lensed Type Ia supernova and measure time delays between the images.

We cover:
1. Loading the model
2. Inspecting the photometry data
3. Fitting with `fit_lensed_sn`
4. Extracting and plotting time-delay posteriors
5. Saving and loading results

## 1. Setup

First, import the necessary packages and initialise the BayeSN model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bayesn_td import SEDmodel

model = SEDmodel(load_model='G26x_model')

## 2. Inspecting the photometry

bayesn-td expects a text file or DataFrame with columns for the observation time, filter, flux or magnitude, uncertainty, and an image identifier. Replace the path below with your own data.

In [ ]:
phot = pd.read_csv('path/to/photometry.txt', sep=r'\s+', comment='#')
print(f'Columns: {list(phot.columns)}')
print(f'Images: {phot["image"].unique()}')
print(f'Filters: {phot["filter"].unique()}')
print(f'Number of observations: {len(phot)}')
phot.head(12)

Let's plot the light curves for each image and band.

In [ ]:
images = phot['image'].unique()
bands = phot['filter'].unique()
colours = {img: f'C{i}' for i, img in enumerate(images)}

fig, axes = plt.subplots(1, len(bands), figsize=(4 * len(bands), 4), sharey=True)
if len(bands) == 1:
    axes = [axes]

for ax, band in zip(axes, bands):
    for img in images:
        mask = (phot['filter'] == band) & (phot['image'] == img)
        sub = phot[mask]
        ax.errorbar(sub['mjd'], sub['mag'], yerr=sub['magerr'],
                    fmt='o', color=colours[img], label=img, capsize=2)
    ax.set_xlabel('MJD')
    ax.set_title(band)
    ax.invert_yaxis()

axes[0].set_ylabel('Magnitude')
axes[0].legend()
plt.tight_layout()
plt.show()

## 3. Fitting with `fit_lensed_sn`

`fit_lensed_sn` loads the data and runs MCMC in a single call. Column names are auto-detected from the file header. For a plain text file, pass the metadata explicitly:

In [ ]:
samples = model.fit_lensed_sn(
    photometry='path/to/photometry.txt',
    peak_mjds=[60000.0, 60050.0, 60080.0],  # ordered by sorted unique image labels
    z=1.5,
    ebv_mw=0.05,
    filt_map={'f150w': 'F150W', 'f200w': 'F200W'},  # if needed
    error_floor={'F150W': 0.06},         # optional per-band error floor
    num_samples=500,
    num_warmup=500,
    num_chains=4,
    outputdir='results/my_fit',
)

## 4. Extracting and plotting time-delay posteriors

The fitted samples dictionary contains `delta_t`, which gives the time delays between image 0 and each subsequent image. The shape is `(chains, samples, N_images-1, SNe)`.

In [ ]:
# Print time delay constraints
delta_t = samples['delta_t']
n_delays = delta_t.shape[2]

for i in range(n_delays):
    dt = delta_t[:, :, i, 0].flatten()
    print(f'Delta_t (image 0 - image {i+1}): '
          f'{np.mean(dt):.2f} +/- {np.std(dt):.2f} days '
          f'(median: {np.median(dt):.2f})')

In [ ]:
# Plot time delay posterior distributions
fig, axes = plt.subplots(1, n_delays, figsize=(5 * n_delays, 4))
if n_delays == 1:
    axes = [axes]

for i in range(n_delays):
    dt = delta_t[:, :, i, 0].flatten()
    axes[i].hist(dt, bins=50, density=True, alpha=0.7, color=f'C{i}')
    axes[i].axvline(np.median(dt), color='k', ls='--',
                    label=f'Median: {np.median(dt):.2f} d')
    axes[i].set_xlabel(f'$\\Delta t$ (image 0 $-$ image {i+1}) [days]')
    axes[i].set_ylabel('Posterior density')
    axes[i].legend()

plt.tight_layout()
plt.show()

### Corner plot of key parameters

A corner plot of the shared SN parameters ($\theta$, $A_V$) and the time delays.

In [ ]:
try:
    import corner

    theta = samples['theta'][:, :, 0].flatten()
    AV = samples['AV'][:, :, 0].flatten()
    cols = [theta, AV]
    labels = [r'$\theta$', r'$A_V$']

    for i in range(n_delays):
        cols.append(delta_t[:, :, i, 0].flatten())
        labels.append(f'$\\Delta t_{{0{i+1}}}$')

    data_corner = np.column_stack(cols)
    fig = corner.corner(data_corner, labels=labels, quantiles=[0.16, 0.5, 0.84],
                        show_titles=True, title_kwargs={'fontsize': 12})
    plt.show()
except ImportError:
    print('Install the corner package for corner plots: pip install corner')

## 5. Saving and loading results

The fitting methods automatically save the results to disk. You can reload them later without re-running the fit.

In [ ]:
import pickle

# Load previously saved chains
with open('results/my_fit/chains.pkl', 'rb') as f:
    loaded_samples = pickle.load(f)

print('Loaded sample keys:', list(loaded_samples.keys()))
print('delta_t shape:', loaded_samples['delta_t'].shape)

In [ ]:
# Load the ArviZ summary
summary = pd.read_csv('results/my_fit/fit_summary.csv', index_col=0)
summary.head(10)